# Donor Churn Classifier
**Lighthouse Sanctuary — ML Pipeline**

Predict whether a donor has lapsed (no donation in 180 days) and rank at-risk donors for re-engagement campaigns.

---
## 1. Problem Framing

### Business Problem
Lighthouse Sanctuary relies on recurring donor revenue to fund safehouse operations across the Philippines. Donor churn — defined as **no donation in the past 180 days** — directly threatens program continuity. The average re-activation cost of a lapsed donor is ~3–5× the cost of retaining an active one.

### ML Task
- **Type:** Binary classification  
- **Target:** `is_churned` (1 = lapsed ≥180 days, 0 = active)  
- **Unit of analysis:** One row per supporter  
- **Success metric:** ROC-AUC ≥ 0.75 on held-out test set  

### Features used
| Feature | Description |
|---|---|
| `days_since_last_donation` | Primary churn signal |
| `total_lifetime_value` | Total PHP donated |
| `donation_frequency` | Avg donations/month |
| `num_campaigns` | Distinct campaigns donated to |
| `acquisition_channel` | How supporter joined |
| `supporter_type` | Individual / Corporate / etc. |
| `is_recurring_donor` | Has set up recurring gift |
| `avg_gift_size` | Average donation (PHP) |

### Models compared
1. Logistic Regression (interpretable baseline)  
2. Random Forest (handles non-linearity)  
3. XGBoost (gradient boosting, typically best performance)  

### Deployment target
- **Output:** Saved `models/donor_churn_model.pkl`  
- **API:** `POST /predict/donor-churn` returns `{churn_probability, risk_label}`

---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import sys
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Paths — works whether notebook is run from ml-pipelines/notebooks/ or repo root
NB_DIR   = Path().resolve()
ML_DIR   = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR / 'ml-pipelines'
DATA_DIR = ML_DIR / 'data'
RAW_DIR  = ML_DIR.parent / 'MLPipelines' / 'Data' / 'lighthouse_v7'
MODEL_DIR = ML_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)

DONOR_MASTER = DATA_DIR / 'donor_master.csv'

# Build master dataset if not present
if not DONOR_MASTER.exists():
    print('Building master datasets...')
    builder = ML_DIR / 'master_dataset_builder.py'
    subprocess.run([sys.executable, str(builder)], check=True)

df = pd.read_csv(DONOR_MASTER, low_memory=False)
print(f'donor_master shape: {df.shape}')
df.head()

In [ ]:
# Dataset overview
print('=== Data Types ===')
print(df.dtypes)
print(f'\nChurn rate: {df["is_churned"].mean():.1%}')
print(f'\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts = df['is_churned'].value_counts()
axes[0].bar(['Active (0)', 'Churned (1)'], churn_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Churn Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 1, f'{v} ({v/len(df):.1%})', ha='center')

# Churn by supporter type
churn_by_type = df.groupby('supporter_type')['is_churned'].mean().sort_values(ascending=True)
churn_by_type.plot(kind='barh', ax=axes[1], color='#e74c3c')
axes[1].set_title('Churn Rate by Supporter Type')
axes[1].set_xlabel('Churn Rate')
axes[1].axvline(df['is_churned'].mean(), color='black', linestyle='--', label='Overall avg')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Numeric feature distributions by churn status
numeric_features = [
    'days_since_last_donation', 'total_lifetime_value',
    'donation_frequency', 'avg_gift_size', 'num_campaigns'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    if col in df.columns:
        for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
            subset = df[df['is_churned'] == label][col].dropna()
            axes[i].hist(subset, bins=30, alpha=0.6, color=color,
                        label='Active' if label == 0 else 'Churned')
        axes[i].set_title(col)
        axes[i].legend()
        axes[i].set_xlabel('Value')

axes[-1].axis('off')
plt.suptitle('Feature Distributions by Churn Status', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap on numeric features
numeric_cols = [
    'is_churned', 'days_since_last_donation', 'total_lifetime_value',
    'donation_frequency', 'avg_gift_size', 'num_campaigns',
    'is_recurring_donor', 'total_donations'
]
existing = [c for c in numeric_cols if c in df.columns]
corr = df[existing].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by acquisition channel
plt.figure(figsize=(10, 4))
if 'acquisition_channel' in df.columns:
    churn_by_channel = df.groupby('acquisition_channel')['is_churned'].agg(['mean', 'count']).reset_index()
    churn_by_channel.columns = ['channel', 'churn_rate', 'count']
    churn_by_channel = churn_by_channel.sort_values('churn_rate')
    
    bars = plt.bar(churn_by_channel['channel'], churn_by_channel['churn_rate'],
                   color=plt.cm.RdYlGn_r(churn_by_channel['churn_rate']))
    plt.axhline(df['is_churned'].mean(), color='navy', linestyle='--', label='Overall avg')
    plt.title('Churn Rate by Acquisition Channel')
    plt.ylabel('Churn Rate')
    plt.xticks(rotation=30, ha='right')
    plt.legend()
    
    for bar, count in zip(bars, churn_by_channel['count']):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'n={count}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## 3. Modeling & Feature Selection

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
import xgboost as xgb
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import joblib

In [ ]:
# Feature engineering
FEATURE_COLS = [
    'days_since_last_donation', 'total_lifetime_value', 'donation_frequency',
    'num_campaigns', 'acquisition_channel', 'supporter_type',
    'is_recurring_donor', 'avg_gift_size'
]
TARGET = 'is_churned'

# Keep only rows with the target and all features present
model_df = df[FEATURE_COLS + [TARGET]].copy()
model_df = model_df.dropna(subset=[TARGET])

# Fill numeric NAs with median
numeric_features = [
    'days_since_last_donation', 'total_lifetime_value', 'donation_frequency',
    'num_campaigns', 'is_recurring_donor', 'avg_gift_size'
]
categorical_features = ['acquisition_channel', 'supporter_type']

for c in numeric_features:
    model_df[c] = model_df[c].fillna(model_df[c].median())
for c in categorical_features:
    model_df[c] = model_df[c].fillna('Unknown')

print(f'Model dataset shape: {model_df.shape}')
print(f'Churn rate: {model_df[TARGET].mean():.1%}')
model_df[FEATURE_COLS].describe()

In [ ]:
# Build preprocessing pipeline
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

X = model_df[FEATURE_COLS]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')

In [ ]:
# Define all three models
models = {
    'Logistic Regression': Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('prep', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=8,
                                        class_weight='balanced', random_state=42, n_jobs=-1))
    ]),
    'XGBoost': Pipeline([
        ('prep', preprocessor),
        ('clf', xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
                                    eval_metric='logloss', random_state=42, n_jobs=-1))
    ]),
}

# Cross-validation comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print('5-fold CV ROC-AUC scores:')
print('-' * 40)
for name, pipeline in models.items():
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:25s}: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Train all models on full training set
fitted_models = {}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    fitted_models[name] = pipeline
    print(f'Trained: {name}')

---
## 4. Evaluation & Interpretation

In [ ]:
from sklearn.metrics import roc_curve, auc, ConfusionMatrixDisplay

# ROC-AUC curves
plt.figure(figsize=(8, 6))

test_aucs = {}
colors = ['#3498db', '#2ecc71', '#e74c3c']

for (name, pipeline), color in zip(fitted_models.items(), colors):
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    test_aucs[name] = roc_auc
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={roc_auc:.3f})')

plt.plot([0,1],[0,1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Donor Churn Classifier')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

print('\nTest ROC-AUC Summary:')
for name, score in test_aucs.items():
    print(f'  {name:25s}: {score:.4f}')

best_model_name = max(test_aucs, key=test_aucs.get)
print(f'\nBest model: {best_model_name}')

In [ ]:
# Detailed evaluation of best model
best_pipeline = fitted_models[best_model_name]
y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]

print(f'Classification Report — {best_model_name}')
print('=' * 60)
print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Active', 'Churned'],
    cmap='Blues', ax=ax
)
ax.set_title(f'Confusion Matrix — {best_model_name}')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from best tree-based model
# Try XGBoost first, fall back to RF
for model_name in ['XGBoost', 'Random Forest']:
    if model_name in fitted_models:
        pipeline = fitted_models[model_name]
        clf = pipeline.named_steps['clf']
        
        all_feature_names = numeric_features + categorical_features
        importances = clf.feature_importances_
        
        feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=True)
        
        plt.figure(figsize=(8, 5))
        feat_imp.plot(kind='barh', color='#3498db')
        plt.title(f'Feature Importances — {model_name}')
        plt.xlabel('Importance')
        plt.tight_layout()
        plt.show()
        
        print(f'Top 3 churn predictors ({model_name}):')
        for feat, imp in feat_imp.sort_values(ascending=False).head(3).items():
            print(f'  {feat}: {imp:.4f}')
        break

In [ ]:
# Logistic Regression coefficients (interpretable)
lr_pipeline = fitted_models['Logistic Regression']
lr_clf = lr_pipeline.named_steps['clf']
all_feature_names = numeric_features + categorical_features

coef_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Coefficient': lr_clf.coef_[0]
}).sort_values('Coefficient')

plt.figure(figsize=(8, 5))
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Logistic Regression Coefficients\n(positive = increases churn probability)')
plt.xlabel('Coefficient')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation comparison boxplot
plt.figure(figsize=(8, 4))
cv_df = pd.DataFrame(cv_results)
cv_df.boxplot(figsize=(8, 4))
plt.title('5-Fold CV ROC-AUC Distribution by Model')
plt.ylabel('ROC-AUC')
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

---
## 5. Causal and Relationship Analysis

In [ ]:
# Causal analysis: does recurring donor status causally protect against churn?
print('=== Recurring Donor vs. Churn ===')
recurring_churn = df.groupby('is_recurring_donor')['is_churned'].agg(['mean','count'])
recurring_churn.index = ['One-time', 'Recurring']
recurring_churn.columns = ['Churn Rate', 'N']
print(recurring_churn.to_string())
print()

# Segmentation: high-value vs. low-value donors
median_ltv = df['total_lifetime_value'].median()
df['value_segment'] = pd.cut(
    df['total_lifetime_value'],
    bins=[0, median_ltv, df['total_lifetime_value'].quantile(0.9), np.inf],
    labels=['Low Value', 'Mid Value', 'High Value']
)

segment_analysis = df.groupby('value_segment').agg(
    churn_rate=('is_churned','mean'),
    avg_days_since_last=('days_since_last_donation','mean'),
    avg_frequency=('donation_frequency','mean'),
    count=('supporter_id','count')
)
print('=== Churn by Donor Value Segment ===')
print(segment_analysis.to_string())

In [ ]:
# Churn probability distribution by risk segment
y_proba_all = best_pipeline.predict_proba(X)[:, 1]
df_risk = df.copy()
df_risk['churn_proba'] = y_proba_all
df_risk['risk_label'] = pd.cut(
    df_risk['churn_proba'],
    bins=[0, 0.4, 0.7, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

risk_dist = df_risk['risk_label'].value_counts()
print('Risk Segment Distribution:')
print(risk_dist.to_string())

plt.figure(figsize=(8, 4))
plt.hist(df_risk['churn_proba'], bins=40, color='#e74c3c', alpha=0.7, edgecolor='white')
plt.axvline(0.4, color='orange', linestyle='--', label='Medium threshold (0.4)')
plt.axvline(0.7, color='red', linestyle='--', label='High threshold (0.7)')
plt.title('Distribution of Predicted Churn Probabilities')
plt.xlabel('Churn Probability')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Relationship: days_since_last_donation vs. churn probability (smoothed)
plt.figure(figsize=(8, 4))

# Bin days into intervals and plot churn rate
bins = list(range(0, 400, 20))
df_tmp = df.copy()
df_tmp['days_bin'] = pd.cut(df_tmp['days_since_last_donation'], bins=bins)
churn_by_days = df_tmp.groupby('days_bin', observed=True)['is_churned'].mean()

churn_by_days.plot(kind='bar', color='#e74c3c', alpha=0.7)
plt.axvline(x=9, color='black', linestyle='--', label='180-day mark (~bin 9)')
plt.title('Actual Churn Rate vs. Days Since Last Donation')
plt.xlabel('Days Since Last Donation (binned)')
plt.ylabel('Churn Rate')
plt.xticks(rotation=60, fontsize=7)
plt.tight_layout()
plt.show()

---
## 6. Deployment Notes

In [ ]:
# Save the best model
model_path = MODEL_DIR / 'donor_churn_model.pkl'
joblib.dump(best_pipeline, model_path)
print(f'Model saved: {model_path}')
print(f'Best model: {best_model_name}')
print(f'Test ROC-AUC: {test_aucs[best_model_name]:.4f}')

### API Usage

Start the FastAPI service from `ml-pipelines/`:
```bash
uvicorn api.main:app --reload --port 8001
```

Call the endpoint:
```bash
curl -X POST http://localhost:8001/predict/donor-churn \
  -H 'Content-Type: application/json' \
  -d '{
    "days_since_last_donation": 250,
    "total_lifetime_value": 5000,
    "donation_frequency": 0.5,
    "num_campaigns": 2,
    "acquisition_channel": "SocialMedia",
    "supporter_type": "Individual",
    "is_recurring_donor": false,
    "avg_gift_size": 2500
  }'
```

**Response:**
```json
{
  "churn_probability": 0.82,
  "risk_label": "High"
}
```

### Monitoring
- Retrain quarterly with fresh donations data
- Alert if mean predicted churn probability shifts by >10% month-over-month
- Track precision/recall at the 0.5 threshold in production

### Integration with .NET Backend
The .NET backend should call this endpoint for every supporter viewed in the admin dashboard, caching results for 24 hours to avoid repeated ML calls.